# 01 — Pair Correlations & Power-Law Fits

Analyse the equal-time pair correlator $P(r)$ for the 1D attractive Hubbard model.

**Research question**: Does $P(r)$ decay as a power law $\sim r^{-\eta_P}$ (Luther-Emery liquid)
or exponentially (gapped)? For $U < 0$ and $n_\sigma = 0.4$ we expect power-law in the NN case;
long-range hopping should reduce $\eta_P$.

**Sections**
1. NN hopping at $n_\sigma = 0.4$ — confirm power-law, extract $\eta_P$
2. LR hopping at $n_\sigma = 0.4$ — vary $\alpha$, measure change in $\eta_P$
3. Power-law exponent summary — compare to Luther-Emery prediction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import sys, os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
from analysis.extract import SimParams, extract_equal_time

plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

DATA_BASE = "/nfs/home/gissa/Hubbard"  # adjust as needed


def power_law_fit(r, P, r_min=5, r_max=None):
    """Fit P(r) = A * r^{-eta} via log-log linear regression."""
    if r_max is None:
        r_max = r.max()
    mask = (r >= r_min) & (r <= r_max) & (P > 0)
    slope, intercept, rval, _, stderr = linregress(np.log(r[mask]), np.log(P[mask]))
    return -slope, np.exp(intercept), rval**2  # eta, A, R^2


def plot_pair_corr(ax, r, P, err, label, color=None):
    """Log-log plot of P(r) with error bars."""
    ax.errorbar(r[r > 0], P[r > 0], yerr=err[r > 0],
                fmt='o', ms=4, capsize=3, label=label, color=color)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$r$')
    ax.set_ylabel(r'$P(r)$')

## Section 1: NN hopping, $n_\sigma = 0.4$

Update `mu_star` after running the density calibration in `00_sanity.ipynb`.

In [ ]:
mu_star = -2.4   # <-- update after density calibration
U = -5.0
L = 100

nn_runs = [
    SimParams(U=U, mu=mu_star, beta=40.0,  L=L, alpha=2.0, Rmax=1),
    SimParams(U=U, mu=mu_star, beta=60.0,  L=L, alpha=2.0, Rmax=1),
    SimParams(U=U, mu=mu_star, beta=80.0,  L=L, alpha=2.0, Rmax=1),
    SimParams(U=U, mu=mu_star, beta=100.0, L=L, alpha=2.0, Rmax=1),
]

fig, ax = plt.subplots(figsize=(7, 5))
eta_nn = {}

for params in nn_runs:
    try:
        data = extract_equal_time(params, 'pair', 'position', base=DATA_BASE)
    except FileNotFoundError as e:
        print(f"[skip] β={params.beta}: {e}")
        continue

    r, P, err = data['r'], data['values'], data['errors']
    eta, A, r2 = power_law_fit(r, P)
    eta_nn[params.beta] = eta
    lbl = f'β={params.beta:.0f}  η={eta:.3f}  (R²={r2:.3f})'
    plot_pair_corr(ax, r, P, err, label=lbl)

    # Overlay power-law fit
    r_fit = np.linspace(5, r.max(), 200)
    ax.plot(r_fit, A * r_fit**(-eta), '--', lw=1)

ax.set_title(f'NN pair correlator  U={U}, L={L}, μ={mu_star}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/pair_corr_NN.pdf')
plt.show()
print('NN exponents:', eta_nn)

## Section 2: LR hopping, $n_\sigma = 0.4$

Update `mu_lr_star` after running the LR density calibration.

In [ ]:
mu_lr_star = -2.4   # <-- update after LR density calibration
beta_lr = 60.0
Rmax_lr = 50

lr_runs = [
    SimParams(U=U, mu=mu_lr_star, beta=beta_lr, L=L, alpha=0.5, Rmax=Rmax_lr),
    SimParams(U=U, mu=mu_lr_star, beta=beta_lr, L=L, alpha=0.8, Rmax=Rmax_lr),
    SimParams(U=U, mu=mu_lr_star, beta=beta_lr, L=L, alpha=1.0, Rmax=Rmax_lr),
    SimParams(U=U, mu=mu_lr_star, beta=beta_lr, L=L, alpha=1.5, Rmax=Rmax_lr),
    SimParams(U=U, mu=mu_lr_star, beta=beta_lr, L=L, alpha=2.0, Rmax=Rmax_lr),
]

fig, ax = plt.subplots(figsize=(7, 5))
eta_lr = {}

for params in lr_runs:
    try:
        data = extract_equal_time(params, 'pair', 'position', base=DATA_BASE)
    except FileNotFoundError as e:
        print(f"[skip] α={params.alpha}: {e}")
        continue

    r, P, err = data['r'], data['values'], data['errors']
    eta, A, r2 = power_law_fit(r, P)
    eta_lr[params.alpha] = eta
    lbl = f'α={params.alpha}  η={eta:.3f}'
    plot_pair_corr(ax, r, P, err, label=lbl)

    r_fit = np.linspace(5, r.max(), 200)
    ax.plot(r_fit, A * r_fit**(-eta), '--', lw=1)

ax.set_title(f'LR pair correlator  U={U}, β={beta_lr}, L={L}, μ={mu_lr_star}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/pair_corr_LR.pdf')
plt.show()
print('LR exponents:', eta_lr)

## Section 3: Power-law exponent summary

Luther-Emery prediction: $\eta_P = 1 / (2 K_\rho)$ where $K_\rho > 1$ for attraction.
LR hopping is expected to enhance pairing → reduce $\eta_P$.

In [ ]:
if eta_lr:
    alphas_sorted = sorted(eta_lr.keys())
    etas_sorted   = [eta_lr[a] for a in alphas_sorted]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(alphas_sorted, etas_sorted, 'o-', label='DQMC')

    # Mark NN value as horizontal reference
    if eta_nn:
        eta_nn_mean = np.mean(list(eta_nn.values()))
        ax.axhline(eta_nn_mean, color='gray', ls='--', label=f'NN avg η={eta_nn_mean:.3f}')

    ax.set_xlabel(r'$\alpha$')
    ax.set_ylabel(r'$\eta_P$')
    ax.set_title(r'Pair correlator exponent vs $\alpha$')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/eta_vs_alpha.pdf')
    plt.show()
else:
    print('No LR data loaded yet.')